# 06 – Merge and Join

So far you've been working with a single DataFrame.

In the real world, data is almost never in one place. You'll have:
- One table with student details
- Another table with their marks
- A third table with department info

To answer any real question, you need to **combine** these tables together.

That's what `merge()` and `join()` do.

If you've used SQL before, this is the same idea as a JOIN. If you haven't — think of it like a VLOOKUP in Excel, but much more powerful.

The key concept: you combine two DataFrames by matching rows that share a **common column** (called the key).

## Setting Up — Two Separate Tables

In [ ]:
import pandas as pd

# Table 1 — student basic info
students = pd.DataFrame({
    "Student_ID": [101, 102, 103, 104, 105],
    "Name":       ["Amit", "Priya", "Ravi", "Neha", "Karan"],
    "City":       ["Pune", "Mumbai", "Delhi", "Pune", "Mumbai"]
})

# Table 2 — student marks
marks = pd.DataFrame({
    "Student_ID": [101, 102, 103, 104, 106],
    "Marks":      [85, 90, 78, 92, 88],
    "Grade":      ["B", "A", "C", "A", "B"]
})

print("Students table:")
print(students)
print()
print("Marks table:")
print(marks)

Notice — the two tables share `Student_ID`. That's the **key** we'll use to match rows.

Also notice something important:
- Student 105 (Karan) is in the students table but NOT in the marks table
- Student 106 is in the marks table but NOT in the students table

How we handle these mismatches is what the different types of joins are about.

## 1) Inner Join — Keep Only Matching Rows

An inner join keeps only the rows where the key exists in **both** tables.

If a row doesn't have a match in the other table, it gets dropped.

In [ ]:
# Inner join — default behavior of merge()

inner = pd.merge(students, marks, on="Student_ID", how="inner")
print(inner)

Only students 101, 102, 103, 104 appear — because they exist in **both** tables.

- Student 105 (Karan) was dropped — no marks record found
- Student 106 was dropped — no student details found

Use inner join when you only care about rows that have complete data on both sides.

## 2) Left Join — Keep Everything from the Left Table

A left join keeps **all rows from the left table**, and matches in data from the right table where it can.

If no match is found on the right, those columns are filled with `NaN`.

In [ ]:
# Left join — keep all students, attach marks where available

left = pd.merge(students, marks, on="Student_ID", how="left")
print(left)

Now Karan (105) is included — but his Marks and Grade are `NaN` because there's no record in the marks table.

Student 106 is still excluded — it only exists in the right table.

This is the most commonly used join. You usually start with your main table on the left and pull in extra info from other tables.

## 3) Right Join — Keep Everything from the Right Table

The opposite of left join — keeps **all rows from the right table**, fills with `NaN` where the left has no match.

In [ ]:
# Right join — keep all marks records, attach student details where available

right = pd.merge(students, marks, on="Student_ID", how="right")
print(right)

Now Student 106 appears — but Name and City are `NaN` since there's no student record for them.

Karan (105) is gone — he only exists in the left table.

Right join is rarely used in practice. Most people just swap the table order and use a left join instead — same result, easier to think about.

## 4) Outer Join — Keep Everything from Both Tables

An outer join (also called full outer join) keeps **all rows from both tables**. Missing values are filled with `NaN` on whichever side has no match.

In [ ]:
# Outer join — keep every row from both tables

outer = pd.merge(students, marks, on="Student_ID", how="outer")
print(outer)

All 6 unique Student IDs appear now — 101 to 106.
- Karan (105) has NaN in Marks/Grade
- Student 106 has NaN in Name/City

Use outer join when you want to see the complete picture — including all mismatches.

## 5) Visual Summary of All Four Joins

```
Left Table (students):   101, 102, 103, 104, 105
Right Table (marks):     101, 102, 103, 104,      106

INNER  →  101, 102, 103, 104              (only matches)
LEFT   →  101, 102, 103, 104, 105         (all from left)
RIGHT  →  101, 102, 103, 104,      106    (all from right)
OUTER  →  101, 102, 103, 104, 105, 106    (everything)
```

In practice:
- **Inner** — use when you only want complete, matched data
- **Left** — use most of the time; keep your main table intact and enrich it
- **Right** — rarely used; just flip the table order and use left
- **Outer** — use when you need to audit mismatches or see the full picture

## 6) Merging on Columns with Different Names

Sometimes the same information has a different column name in each table. You can still merge using `left_on` and `right_on`.

In [ ]:
# Same data but the key column has different names in each table

students2 = pd.DataFrame({
    "ID":   [101, 102, 103, 104],
    "Name": ["Amit", "Priya", "Ravi", "Neha"]
})

marks2 = pd.DataFrame({
    "Stu_ID": [101, 102, 103, 104],
    "Marks":  [85, 90, 78, 92]
})

# left_on = key column name in the left table
# right_on = key column name in the right table
result = pd.merge(students2, marks2, left_on="ID", right_on="Stu_ID")
print(result)

Both key columns appear in the result (`ID` and `Stu_ID`). You can drop the duplicate one afterwards.

In [ ]:
# Drop the duplicate key column

result = result.drop("Stu_ID", axis=1)
print(result)

## 7) Merging on Multiple Keys

Sometimes you need more than one column to uniquely identify a row — like Name + Date, or City + Department.

In [ ]:
# Exam scores across two subjects — same student can appear twice

scores1 = pd.DataFrame({
    "Name":    ["Amit", "Amit", "Priya", "Priya"],
    "Subject": ["Math", "Science", "Math", "Science"],
    "Score":   [85, 78, 90, 88]
})

scores2 = pd.DataFrame({
    "Name":    ["Amit", "Amit", "Priya", "Priya"],
    "Subject": ["Math", "Science", "Math", "Science"],
    "Grade":   ["B", "C", "A", "B"]
})

# Merge on both Name AND Subject together
result = pd.merge(scores1, scores2, on=["Name", "Subject"])
print(result)

When a single column isn't enough to uniquely identify a row, pass a list of column names to `on`.

## 8) Concatenating DataFrames with `concat()`

Merging combines tables **side by side** (matching columns).

`concat()` stacks DataFrames **on top of each other** (adding rows).

Use this when you have the same structure of data split across multiple files — like monthly sales reports.

In [ ]:
# Two batches of students — same columns, different rows

batch1 = pd.DataFrame({
    "Name":  ["Amit", "Priya", "Ravi"],
    "Marks": [85, 90, 78]
})

batch2 = pd.DataFrame({
    "Name":  ["Neha", "Karan", "Sneha"],
    "Marks": [92, 88, 74]
})

# Stack them vertically
all_students = pd.concat([batch1, batch2])
print(all_students)

Notice the index — it goes 0, 1, 2, then 0, 1, 2 again. The original indexes were kept.

Always reset the index after concatenating.

In [ ]:
# Clean index after concat

all_students = pd.concat([batch1, batch2], ignore_index=True)
print(all_students)

`ignore_index=True` resets the index automatically — no need to call `reset_index()` separately.

In [ ]:
# concat() also works with more than two DataFrames

batch3 = pd.DataFrame({
    "Name":  ["Raj", "Anita"],
    "Marks": [81, 95]
})

all_students = pd.concat([batch1, batch2, batch3], ignore_index=True)
print(all_students)

## 9) Putting It All Together — A Realistic Workflow

You have three separate tables. Let's combine them into one complete student report.

In [ ]:
# Table 1 — student info
students = pd.DataFrame({
    "Student_ID": [101, 102, 103, 104, 105],
    "Name":       ["Amit", "Priya", "Ravi", "Neha", "Karan"],
    "Dept_ID":    [1, 2, 1, 3, 2]
})

# Table 2 — exam marks
marks = pd.DataFrame({
    "Student_ID": [101, 102, 103, 104, 105],
    "Marks":      [85, 90, 78, 92, 88]
})

# Table 3 — department names
departments = pd.DataFrame({
    "Dept_ID":    [1, 2, 3],
    "Department": ["Science", "Arts", "Commerce"]
})

# Step 1 — merge students with marks
df = pd.merge(students, marks, on="Student_ID", how="left")

# Step 2 — merge the result with departments
df = pd.merge(df, departments, on="Dept_ID", how="left")

# Step 3 — drop the Dept_ID column since we now have the name
df = df.drop("Dept_ID", axis=1)

# Step 4 — sort by Marks
df = df.sort_values("Marks", ascending=False).reset_index(drop=True)

print(df)

Three separate tables, two merges, and a clean final result. This is a very common pattern in real data work — data is stored in normalized tables and you bring them together as needed.

## Summary

**merge() — combine side by side by matching a key column:**
- `how="inner"` — only rows that exist in both tables
- `how="left"` — all rows from the left, match from right where possible
- `how="right"` — all rows from the right, match from left where possible
- `how="outer"` — all rows from both tables
- `on="col"` — key column (same name in both tables)
- `left_on=` / `right_on=` — when the key has different names in each table
- `on=["col1", "col2"]` — merge on multiple keys

**concat() — stack rows on top of each other:**
- `pd.concat([df1, df2], ignore_index=True)` — combine multiple DataFrames vertically

Next up: **Working with Dates in Pandas** — parsing, extracting, and filtering by date.

## Practice Questions 1

1. Create two DataFrames — `employees` (Employee_ID, Name, Dept_ID) and `salaries` (Employee_ID, Salary).
2. Do a left join to combine them.
3. Do an inner join and compare the result — do any rows differ?

## Practice Questions 2

1. Create a third table `departments` (Dept_ID, Department_Name).
2. Merge all three tables together into one complete employee report.
3. Create two monthly sales DataFrames (Jan, Feb) with the same columns and stack them using `concat()`.